# 003: Loss Function Engineering — MSE, BCE, and Focal Loss derivations

## 1. MEAN SQUARED ERROR (MSE)
- Used for regression tasks:
  $$\mathcal{L}_{\text{MSE}} = \frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

## 2. BINARY CROSS-ENTROPY (BCE)
- Used for binary classification tasks:
  $$\mathcal{L}_{\text{BCE}} = -\frac{1}{m} \sum_{i=1}^{m} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

## 3. FOCAL LOSS (Lin et al., 2017)
- Addresses class imbalance (e.g., in target detection) by down-weighting easy examples:
  $$\mathcal{L}_{\text{Focal}} = -\frac{1}{m} \sum_{i=1}^{m} \alpha_t (1 - p_t)^\gamma \log(p_t)$$
  where $p_t = \hat{y}_i$ if $y_i=1$, else $1 - \hat{y}_i$.
  - $\gamma$ (focusing parameter) dynamically scales back the loss of easy-to-predict samples ($\gamma = 2$ is standard).
---


In [ ]:
import numpy as np

class LossEngineering:
    """Implementations of key loss functions with gradient helpers."""
    @staticmethod
    def mse(y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    @staticmethod
    def binary_cross_entropy(y_true, y_pred, eps=1e-15):
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    @staticmethod
    def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25, eps=1e-15):
        """Focal Loss from scratch for binary targets."""
        y_pred = np.clip(y_pred, eps, 1 - eps)
        
        # Calculate pt
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        # Calculate alpha weight
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        
        loss = -alpha_t * ((1 - p_t) ** gamma) * np.log(p_t)
        return np.mean(loss)



In [ ]:
# Testing the loss metrics
print("--- Loss Value Comparisons ---")
y_true = np.array([1, 0, 1, 1, 0])

# Scenario A: Highly confident correct predictions
y_pred_correct = np.array([0.95, 0.05, 0.99, 0.90, 0.10])
print(f"Correct predictions:")
print(f"  BCE Loss:   {LossEngineering.binary_cross_entropy(y_true, y_pred_correct):.6f}")
print(f"  Focal Loss: {LossEngineering.focal_loss(y_true, y_pred_correct, gamma=2.0):.6f}")

# Scenario B: Confident but incorrect predictions (hard/misclassified examples)
y_pred_incorrect = np.array([0.10, 0.90, 0.05, 0.20, 0.85])
print(f"\nIncorrect predictions:")
print(f"  BCE Loss:   {LossEngineering.binary_cross_entropy(y_true, y_pred_incorrect):.6f}")
print(f"  Focal Loss: {LossEngineering.focal_loss(y_true, y_pred_incorrect, gamma=2.0):.6f}")

print("\n[Insight] Focal Loss dampens losses for correct (easy) predictions significantly, focusing on incorrect ones.")
